# 04 — Train YOLO26m

Pipeline Ultralytics YOLO26m trên VNTS, 52 lớp, không crop từng biển. Checkpoint chọn theo mAP50-95 trên val. Mosaic và mixup chỉ chạy khi `ONLINE = True`.


## 0) Bootstrap (Kaggle / Colab)

Clone hoặc pull nhánh, rồi cài dependency. Trên Kaggle bật Internet: On.


In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/tuan8p/VN-Traffic-Sign-Detection.git"
REPO_DIR = "VN-Traffic-Sign-Detection"
REPO_BRANCH = "yolo"  # Team YOLO đổi thành yolo. Team Faster R-CNN đổi thành faster-rcnn khi nhánh đã có trên origin.

base = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
repo_path = base / REPO_DIR

def sh(cmd: str, check: bool = True) -> int:
    print(">>", cmd)
    return subprocess.run(cmd, shell=True, check=check).returncode

if (Path.cwd() / ".git").exists() and (Path.cwd() / "configs" / "shared.yaml").exists():
    repo_path = Path.cwd()
    print("Using the repo this notebook sits in:", repo_path)
elif not (repo_path / ".git").exists():
    sh(f'git clone --branch {REPO_BRANCH} {REPO_URL} "{repo_path}"')

git = f'git -C "{repo_path}"'
sh(f"{git} fetch origin {REPO_BRANCH}")
sh(f"{git} checkout -B {REPO_BRANCH} origin/{REPO_BRANCH}")

os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))
print("cwd:", Path.cwd())
sh(f"{git} log --oneline -1")

sh('pip install -q -e ".[detect]"')
sh("pip install -q wandb")

import torch
import vn_tsd
from vn_tsd.config.resolve import resolve_config
print(
    f"Bootstrap OK. PyTorch {torch.__version__} | CUDA: {torch.cuda.is_available()} | GPUs: {torch.cuda.device_count()}"
)


## 1) Đường dẫn data/processed

Cache offline dùng chung: ảnh letterbox và nhãn YOLO, không crop từng biển. Trên Kaggle gắn dataset bằng Add Input rồi điền vào `CANDIDATES`.


In [ ]:
from pathlib import Path

# After Add Input on Kaggle, paste the dataset path here.
CANDIDATES = [
    "/kaggle/input/datasets/zero001206/vn-traffic-sign-detection-preprocessed-data/processed",
    #"data/processed",
]

def _ready(path: Path) -> bool:
    return (path / "data.yaml").exists() or (path / "images" / "train").exists()

PROCESSED_ROOT = next((c for c in CANDIDATES if c and _ready(Path(c))), None)
if PROCESSED_ROOT is None:
    print("[!] data/processed chưa có data.yaml hoặc images/train.")
    print("    Chạy preprocessing offline trước, hoặc gắn dataset Kaggle vào CANDIDATES.")
    PROCESSED_ROOT = "data/processed"
else:
    print("[OK] Processed root:", PROCESSED_ROOT)


## 2) Cấu hình

Sửa biến ở đây. Batch là tổng của cả hai GPU, không phải mỗi GPU. Không lật ngang.


In [ ]:
# Batch train và eval batch là tổng hai GPU, và phải giống nhau ở cả hai pipeline.
# Chọn số sát VRAM của 2 T4. Hai pipeline phải dùng cùng tổng.
# Đổi batch, eval batch hoặc early_stopping_patience thì báo team kia, rồi sửa cả notebook còn lại và configs/shared.yaml.
# Các biến ở đây chỉ là mức cơ bản. Khi build pipeline, team có thể thêm biến khác cho dễ tune.
PIPELINE = "yolo26m"
EPOCHS = 50
LR0 = 0.01
BATCH_SIZE = 8
EARLY_STOPPING_PATIENCE = 10
EVAL_BATCH_SIZE = 16
WEIGHTS = "yolo26m.pt"
IMGSZ = 640
ONLINE = False          # True chỉ khi đang tune online preprocess
MOSAIC = True           # chỉ áp dụng khi ONLINE = True
MIXUP = False
REQUIRE_WANDB = True
USE_AMP = True
PRIMARY_METRIC = "mAP50-95"
SEED = 42
# Tên hiện trên W&B: {pipeline}_s{seed}_ep{epochs}_bs{batch}[_online]_{hash8}
suffix = "_online" if ONLINE else ""
import uuid
RUN_NAME = f"{PIPELINE}_s{SEED}_ep{EPOCHS}_bs{BATCH_SIZE}{suffix}_{uuid.uuid4().hex[:8]}"


## 3) Resolve config

Ghép biến ở trên với `configs/shared.yaml` và file pipeline. Ghi file overrides để lệnh torchrun bên dưới dùng đúng cấu hình này.


In [ ]:
from pathlib import Path
OVERRIDES = {
    "seed": SEED,
    "model": {"weights": WEIGHTS, "imgsz": IMGSZ},
    "train": {
        "epochs": EPOCHS,
        "lr0": LR0,
        "batch_size": BATCH_SIZE,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "amp": USE_AMP,
        "require_wandb": REQUIRE_WANDB,
        "run_name": RUN_NAME,
        "primary_metric": PRIMARY_METRIC,
    },
    "eval": {"batch_size": EVAL_BATCH_SIZE, "auto_eval_test": True, "split": "test"},
    "online_preprocess": {"enabled": ONLINE, "mosaic": MOSAIC, "mixup": MIXUP, "hflip": False},
    "data": {"root": PROCESSED_ROOT},
}


from vn_tsd.config.resolve import resolve_config
from vn_tsd.utils.io import save_yaml

runtime_yaml = "configs/runtime/kaggle.yaml" if Path("/kaggle/working").exists() else "configs/runtime/local.yaml"
cfg = resolve_config(
    pipeline_yaml=f"configs/pipelines/{PIPELINE}.yaml",
    shared_yaml="configs/shared.yaml",
    runtime_yaml=runtime_yaml,
    overrides=OVERRIDES,
)

OVERRIDES_PATH = Path("outputs") / "_notebook" / f"overrides_{PIPELINE}.yaml"
save_yaml(OVERRIDES, OVERRIDES_PATH)

train = cfg.get("train", {})
print("=== Resolved configuration ===")
print("Run name        :", RUN_NAME)
print("Pipeline        :", cfg.get("pipeline"))
print("Epochs          :", train.get("epochs"))
print("Primary metric  :", train.get("primary_metric"))
print("Batch total     :", train.get("batch_size"))
print("Eval batch total:", cfg.get("eval", {}).get("batch_size"))
print("AMP             :", train.get("amp"))
print("W&B required    :", train.get("require_wandb"))
print("Online          :", cfg.get("online_preprocess", {}).get("enabled"))
print("Data root       :", cfg.get("data", {}).get("root"))
print("Overrides file  :", OVERRIDES_PATH)


## 4) W&B Secret
Notebook chỉ đọc Secret. Rank 0 của script khởi tạo và đóng run training.
Không gọi wandb.init() trong notebook. REQUIRE_WANDB=False dùng cho smoke không tracking.

In [ ]:
import os
if REQUIRE_WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY').strip()
    os.environ['WANDB_MODE'] = 'online'
    print('Secret ready; tracking starts inside rank 0.')

## 5) Dataset và launch DDP hai GPU
Cell bên dưới copy thật dataset sang Working để có thể lưu label cache; không dùng symlink. Cell launch chạy torchrun, không train trực tiếp
trong notebook. Đặt EPOCHS=1 ở mục 2 để smoke trước. Batch là tổng hai GPU.
EVALUATE_TEST=False khi tune; chỉ bật test sau khi chốt model. Test chạy một GPU
sau khi các worker kết thúc; không phải evaluation phân tán như R-CNN.
W&B training charts và history ghi từ rank 0. Test có file JSON riêng, chưa tự gửi
vào W&B run đã đóng. AR100 vẫn null. Cần Kaggle T4 x2 hoặc hai CUDA GPU khả dụng.

In [ ]:
from pathlib import Path
import shutil
import uuid
from vn_tsd.utils.io import load_yaml, save_yaml

# Real copy: symlinks still read Input and cannot persist Ultralytics label cache.
source_root = Path(PROCESSED_ROOT).resolve()
if not (source_root / 'data.yaml').is_file():
    raise FileNotFoundError(source_root / 'data.yaml')
for split in ('train', 'val', 'test'):
    for kind in ('images', 'labels'):
        if not (source_root / kind / split).is_dir():
            raise FileNotFoundError(source_root / kind / split)

working = Path('/kaggle/working')
size = sum(p.stat().st_size for p in source_root.rglob('*') if p.is_file())
if shutil.disk_usage(working).free < size * 1.2:
    raise RuntimeError('Not enough space to copy dataset to Working')
# Fresh directory per preparation; do not mix stale cache or overwrite user files.
fixed_root = working / f'yolo_dataset_{uuid.uuid4().hex[:8]}'
shutil.copytree(source_root, fixed_root, ignore=shutil.ignore_patterns('*.cache'))
data_cfg = load_yaml(fixed_root / 'data.yaml')
data_cfg.update(path=str(fixed_root), train='images/train', val='images/val', test='images/test')
save_yaml(data_cfg, fixed_root / 'data.yaml')
PROCESSED_ROOT = str(fixed_root)
print('Writable dataset copy:', PROCESSED_ROOT)
probe = fixed_root / 'labels' / '.write_check'
probe.write_text('ok')
probe.unlink()
print('Label cache directory is writable.')


In [ ]:
import os
import sys
import subprocess
import torch
from vn_tsd.runtime.run_dir import make_run_dir
from vn_tsd.runtime.pack import zip_run_dir
from vn_tsd.utils.io import save_yaml

EVALUATE_TEST = False  # bật sau khi chọn xong config/model
assert torch.cuda.device_count() >= 2, 'Chọn accelerator có hai GPU.'
assert cfg['train']['batch_size'] % 2 == 0
# One identity created in parent; all workers receive the same values.
RUN_DIR = make_run_dir('/kaggle/working/outputs', PIPELINE, RUN_NAME)
OVERRIDES['data']['root'] = PROCESSED_ROOT
OVERRIDES['train']['run_name'] = RUN_NAME
OVERRIDES['eval']['auto_eval_test'] = False
OVERRIDES['runtime_cfg'] = {'device': 'cuda', 'gpus': 2}
OVERRIDES_PATH = RUN_DIR / 'launch_overrides.yaml'
save_yaml(OVERRIDES, OVERRIDES_PATH)
cmd = [sys.executable, '-m', 'torch.distributed.run', '--standalone',
       '--nproc_per_node=2', '-m', 'tools.run_train',
       '--pipeline', PIPELINE, '--runtime', 'configs/runtime/kaggle.yaml',
       '--run-name', RUN_NAME, '--run-dir', str(RUN_DIR),
       '--overrides', str(OVERRIDES_PATH), '--data-root', PROCESSED_ROOT]
print(' '.join(cmd))
subprocess.run(cmd, check=True)
if EVALUATE_TEST:
    subprocess.run([sys.executable, '-m', 'tools.run_eval',
                    '--pipeline', PIPELINE, '--run-dir', str(RUN_DIR)], check=True)
print('ZIP:', zip_run_dir(RUN_DIR))
print('W&B URL: see rank-zero output or', RUN_DIR / 'wandb_run.json')